In [1]:
import pandas as pd
df=pd.read_csv("income.csv")
df

,age,fnlwgt,education_num,capital_gain,capital_loss,hours_per_week,income_level
0,39,77516,13,2174,0,40,0
1,50,83311,13,0,0,13,0
2,38,215646,9,0,0,40,0
3,53,234721,7,0,0,40,0
4,28,338409,13,0,0,40,0
...,...,...,...,...,...,...,...
48837,39,215419,13,0,0,36,0
48838,64,321403,9,0,0,40,0
48839,38,374983,13,0,0,50,0
48840,44,83891,13,5455,0,40,0


### Data Preprocessing

Before building the model, we need to preprocess the data. This involves:
1. Identifying features (X) and the target variable (y).
2. Encoding categorical features, as machine learning models typically require numerical input.
3. Splitting the dataset into training and testing sets to evaluate the model's performance on unseen data.

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Define features (X) and target (y)
X = df.drop('income_level', axis=1)
y = df['income_level']

# Identify categorical columns for encoding
# We'll assume 'education_num', 'capital_gain', 'capital_loss', 'hours_per_week', 'age', 'fnlwgt' are numerical.
categorical_cols = X.select_dtypes(include=['object']).columns

# Apply Label Encoding to categorical features
for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print("Data preprocessing complete. Shapes of training and testing sets:")
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")

Data preprocessing complete. Shapes of training and testing sets:
X_train: (34189, 6), y_train: (34189,)
X_test: (14653, 6), y_test: (14653,)


### Build an AdaBoost Classifier with `n_estimators=10`

Now, let's build an AdaBoost classifier using a `DecisionTreeClassifier` as the base estimator (which is the default) and set `n_estimators` to 10. We will then train the model and measure its prediction score on the test set.

In [3]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Initialize the AdaBoost classifier with n_estimators=10
adaboost_clf_initial = AdaBoostClassifier(n_estimators=10, random_state=42)

# Train the model
adaboost_clf_initial.fit(X_train, y_train)

# Make predictions on the test set
y_pred_initial = adaboost_clf_initial.predict(X_test)

# Calculate the accuracy score
initial_accuracy = accuracy_score(y_test, y_pred_initial)

print(f"AdaBoost Classifier (n_estimators=10) Accuracy: {initial_accuracy:.4f}")

AdaBoost Classifier (n_estimators=10) Accuracy: 0.8277


### Fine-tuning `n_estimators`

To find the best prediction score, we will fine-tune the `n_estimators` parameter by testing a range of values. We'll iterate through different numbers of estimators and record the accuracy for each to identify the optimal value.

In [4]:
import numpy as np

n_estimators_range = np.arange(1, 101, 5) # Test n_estimators from 1 to 100 with steps of 5
scores = []

best_score = 0
best_n_estimators = 0

for n in n_estimators_range:
    adaboost_clf = AdaBoostClassifier(n_estimators=n, random_state=42)
    adaboost_clf.fit(X_train, y_train)
    y_pred = adaboost_clf.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    scores.append(accuracy)

    if accuracy > best_score:
        best_score = accuracy
        best_n_estimators = n

print(f"Best accuracy achieved: {best_score:.4f}")
print(f"Optimal n_estimators: {best_n_estimators}")

Best accuracy achieved: 0.8308
Optimal n_estimators: 96
